In [1]:
import numpy as np
from sklearn.model_selection import GroupKFold

In [2]:
X = np.array([[1, 2], [3, 4], [5, 6], [7, 8], [9, 10], [11, 12]])
y = np.array([1, 2, 3, 4, 5, 6])

In [3]:
X

array([[ 1,  2],
       [ 3,  4],
       [ 5,  6],
       [ 7,  8],
       [ 9, 10],
       [11, 12]])

In [4]:
X.shape

(6, 2)

In [5]:
y.shape

(6,)

In [6]:
groups = np.array([0, 0, 2, 2, 3, 3])

In [11]:
group_kfold = GroupKFold(n_splits=2)

In [12]:
for i, (train_index, test_index) in enumerate(group_kfold.split(X, y, groups)):
    print(F"Fold {i}")
    print(f" Train: index={train_index}, group={groups[train_index]}")
    print(f" Test: index={test_index}, group={groups[test_index]}")

Fold 0
 Train: index=[2 3], group=[2 2]
 Test: index=[0 1 4 5], group=[0 0 3 3]
Fold 1
 Train: index=[0 1 4 5], group=[0 0 3 3]
 Test: index=[2 3], group=[2 2]


number of splits cannot be greater than thew number of groups

In [13]:
for i, (train_index, test_index) in enumerate(group_kfold.split(np.zeros(len(y)), y, groups)):
    print(F"Fold {i}")
    print(f" Train: index={train_index}, group={groups[train_index]}")
    print(f" Test: index={test_index}, group={groups[test_index]}")

Fold 0
 Train: index=[2 3], group=[2 2]
 Test: index=[0 1 4 5], group=[0 0 3 3]
Fold 1
 Train: index=[0 1 4 5], group=[0 0 3 3]
 Test: index=[2 3], group=[2 2]


**using group_kfold with torch datasets:**

In [ ]:
group_kfold = GroupKFold(n_splits=5)

fold_results = []

for fold, (train_indices, val_indices) in enumerate(
    group_kfold.split(
        X=np.zeros(len(y)),
        y=y.numpy(),
        groups=linkbox_ids
    ),
    start=1
):
    print(f"\nFold {fold}")

    train_dataset = Subset(
        full_dataset,
        train_indices
    )

    val_dataset = Subset(
        full_dataset,
        val_indices
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=32,
        shuffle=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=64,
        shuffle=False
    )

    # Crucial: completely new model for every fold
    model = LinkboxClassifier().to(device)

    criterion = torch.nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=3e-4,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=3
    )

    history = train_one_fold(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        scheduler=scheduler,
        device=device,
        epochs=50
    )

    fold_results.append(
        history["best_val_metric"]
    )

then for each fold report:

In [ ]:
fold_results = np.asarray(fold_results)

print("Fold results:", fold_results)
print("Mean:", fold_results.mean())
print("Standard deviation:", fold_results.std())

**for use later: explicitly computing the relationship between 2 sides and enriching the signal:**

Channels 0–2: meters on side A
Channels 3–5: meters on side B

In [ ]:
side_a = x[:, :3, :]
side_b = x[:, 3:, :]

mean_a = side_a.mean(dim=1, keepdim=True)
mean_b = side_b.mean(dim=1, keepdim=True)

mean_difference = mean_a - mean_b
absolute_difference = torch.abs(mean_difference)

std_a = side_a.std(dim=1, keepdim=True)
std_b = side_b.std(dim=1, keepdim=True)

In [ ]:
x_enriched = torch.cat(
    [
        side_a,
        side_b,
        mean_a,
        mean_b,
        mean_difference,
        absolute_difference,
        std_a,
        std_b
    ],
    dim=1
)

**for use later: a two branch model may fit the physical problem better**

Side A: [batch, 3, time]
Side B: [batch, 3, time]

In [ ]:
embedding_a = encoder(side_a)
embedding_b = encoder(side_b)

In [ ]:
comparison = torch.cat(
    [
        embedding_a,
        embedding_b,
        torch.abs(embedding_a - embedding_b),
        embedding_a * embedding_b
    ],
    dim=1
)

logit = classifier(comparison)

In [ ]:
import torch
from torch import nn


class SideEncoder(nn.Module):
    def __init__(self, embedding_dim: int = 32):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(3, 16, kernel_size=7, padding=3),
            nn.GroupNorm(4, 16),
            nn.ReLU(),

            nn.Conv1d(16, 32, kernel_size=5, padding=2),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(64, embedding_dim)
        )

    def forward(self, x):
        return self.network(x)


class LinkboxClassifier(nn.Module):
    def __init__(self, embedding_dim: int = 32):
        super().__init__()

        self.encoder = SideEncoder(embedding_dim)

        self.classifier = nn.Sequential(
            nn.Linear(embedding_dim * 4, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        side_a = x[:, :3, :]
        side_b = x[:, 3:, :]

        embedding_a = self.encoder(side_a)
        embedding_b = self.encoder(side_b)

        comparison = torch.cat(
            [
                embedding_a,
                embedding_b,
                torch.abs(embedding_a - embedding_b),
                embedding_a * embedding_b
            ],
            dim=1
        )

        return self.classifier(comparison).squeeze(1)

**early fusion v. late fusion**

In [ ]:
# recommended approach
features_a = encoder(side_a)
features_b = encoder(side_b)

In [ ]:
combined = torch.cat(
    [
        features_a,
        features_b,
        torch.abs(features_a - features_b),
        features_a * features_b
    ],
    dim=1
)

In [ ]:
joint_features = joint_encoder(combined)
prediction = classifier(joint_features)

In [ ]:
import torch
from torch import nn


class SideEncoder(nn.Module):
    def __init__(self, meters_per_side: int):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(
                meters_per_side,
                16,
                kernel_size=7,
                padding=3
            ),
            nn.GroupNorm(4, 16),
            nn.ReLU(),

            nn.Conv1d(
                16,
                32,
                kernel_size=5,
                padding=2
            ),
            nn.GroupNorm(8, 32),
            nn.ReLU(),

            nn.MaxPool1d(2)
        )

    def forward(self, x):
        return self.network(x)


class LinkboxComparisonCNN(nn.Module):
    def __init__(self, meters_per_side: int = 3):
        super().__init__()

        # Shared weights because both sides contain the same type of signal
        self.side_encoder = SideEncoder(meters_per_side)

        self.joint_encoder = nn.Sequential(
            # features_a, features_b, abs difference and product
            nn.Conv1d(
                32 * 4,
                64,
                kernel_size=5,
                padding=2
            ),
            nn.GroupNorm(8, 64),
            nn.ReLU(),

            nn.Conv1d(
                64,
                64,
                kernel_size=3,
                padding=1
            ),
            nn.GroupNorm(8, 64),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten()
        )

        self.classifier = nn.Sequential(
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        side_a = x[:, :3, :]
        side_b = x[:, 3:, :]

        features_a = self.side_encoder(side_a)
        features_b = self.side_encoder(side_b)

        combined = torch.cat(
            [
                features_a,
                features_b,
                torch.abs(features_a - features_b),
                features_a * features_b
            ],
            dim=1
        )

        features = self.joint_encoder(combined)
        return self.classifier(features).squeeze(1)

test these three:

- Early fusion: all meter channels into one Conv1D.
- Hybrid two-branch: shallow side encoders followed by feature-map comparison.
- Feature baseline: side means, differences and correlations passed to XGBoost or logistic regression.

**tip: using a smaller network:**

class BasicLinkboxCNN(nn.Module):
    def __init__(self, in_channels: int = 6):
        super().__init__()

        self.network = nn.Sequential(
            nn.Conv1d(in_channels, 16, 7, padding=3),
            nn.GroupNorm(4, 16),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(16, 32, 5, padding=2),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, 3, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.network(x).squeeze(1)